# Initialization

In [135]:
import hashlib
import json
from pathlib import Path
import requests

In [136]:
api_key = "Ca8zZF7INWGxJHX14VfgOkl2dm93s0ib"

In [137]:
def search_articles(
        api_key=api_key,
        title_keywords=None,
        keywords=None,
        en=False,
        doi_exists=False,
        union=False,
        limit=5,
        offset=0,
):
    if title_keywords is None:
        title_keywords = []
    if keywords is None:
        keywords = []
    parts = [f'title:"{kw}"' for kw in title_keywords]
    parts += [f'{kw}' for kw in keywords]
    if union == True:
        parts = [f"({" OR ".join(parts)})"]
    if en:
        parts.append("language.code:en")
    if doi_exists:
        parts.append("_exists_:doi")
        
    url = "https://api.core.ac.uk/v3/search/works"
    headers = {"Authorization": f"Bearer {api_key}"}
    params = {
        "q": " AND ".join(parts),
        "limit": limit,
        "offset": offset,
    }
    print(params)
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        response = response.json()
        print("Total hits", response["totalHits"])
        return response
    else:
        print(f"Error: {response.status_code}")
        return None

In [138]:
def filter_articles(
		articles,
		keep_empty=False,
		research=False,
		document_types=None,
):
	if research == True:
		document_types = {"review", "research article", "journal article"}

	texts = {}
	for result in articles["results"]:
		if not keep_empty and len(result["fullText"]) == 0:
			continue
		if document_types is not None:
			if result["documentType"] not in document_types:
				continue
		texts[result["title"]] = result["fullText"]
	return texts

In [139]:
def save_articles(
		articles,
		folder,
):
	folder_path = Path(folder)
	folder_path.mkdir(parents=True, exist_ok=True)
	for text in articles.values():
		hash_code = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]
		file_path = folder_path / (hash_code + ".txt")
		with open(file_path, "w", encoding="utf-8", errors="replace") as f:
			f.write(text)		

# Test

In [122]:
response = search_articles(keywords=["immigration", "immigrants"], union=True, limit=500)

{'q': '(immigration OR immigrants)', 'limit': 500, 'offset': 0}
Total hits 369506


In [140]:
articles = filter_articles(response, keep_empty=False, research=True)

In [141]:
save_articles(articles, "out")

In [ ]:
with open("out/immigration.json", "w") as f:
	json.dump(articles, f, indent=4)